# GRPO post-training for Python code generation

This notebook reproduces the `code-grpo-humaneval` experiment on a hosted Colab GPU. It measures a frozen baseline, runs a one-step GRPO smoke test, optionally trains a LoRA adapter, and evaluates the adapter on a held-out HumanEval split.

**Security:** generated Python runs inside the disposable Colab VM. Do not mount Google Drive and do not add API keys while evaluation or training is running.

## 1. Confirm the hosted GPU
Choose **Runtime > Change runtime type > GPU** before running this cell. A T4 (16 GB) or better is recommended.

In [ ]:
!nvidia-smi
import torch

assert torch.cuda.is_available(), 'No GPU detected. Select a GPU runtime and reconnect.'
props = torch.cuda.get_device_properties(0)
print(f'GPU: {props.name} | VRAM: {props.total_memory / 2**30:.1f} GB')

## 2. Clone the public repository

In [ ]:
import os
import subprocess
from pathlib import Path

repo = Path('/content/code-grpo-humaneval')
if repo.exists():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
else:
    clone_url = 'https://github.com/Hamza-Nadif/code-grpo-humaneval.git'
    subprocess.run(['git', 'clone', clone_url, str(repo)], check=True)
os.chdir(repo)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

## 3. Install and verify dependencies
This installation is stored only in the temporary Colab runtime.

In [ ]:
%pip install -q -r requirements.txt -r requirements-dev.txt

In [ ]:
import importlib.metadata as metadata

for package in ['torch', 'transformers', 'datasets', 'trl', 'peft', 'bitsandbytes']:
    print(f'{package}: {metadata.version(package)}')
!pytest
!ruff check .

## 4. Build deterministic HumanEval splits
The script downloads only the small dataset from a pinned OpenAI commit and writes 120 training, 22 validation, and 22 held-out test tasks.

In [ ]:
!python build_training_data.py --output-dir data
!cat data/manifest.json | head -n 20

## 5. Validate the execution harness
Canonical solutions should obtain pass@1 = 1.0. This is a pipeline check, not a model result.

In [ ]:
!python evaluate_baseline.py \
  --data data/humaneval_test.jsonl \
  --backend oracle \
  --executor local \
  --allow-local-code-execution \
  --output-dir results/oracle-smoke

## 6. Measure the frozen baseline
This downloads `Qwen2.5-Coder-0.5B-Instruct` and evaluates one deterministic completion on each of the 22 held-out tasks. The result is the before-GRPO score.

In [ ]:
!python evaluate_baseline.py \
  --data data/humaneval_test.jsonl \
  --backend transformers \
  --model Qwen/Qwen2.5-Coder-0.5B-Instruct \
  --quantization 4bit \
  --samples-per-task 1 \
  --temperature 0 \
  --executor local \
  --allow-local-code-execution \
  --output-dir results/baseline-heldout

## 7. One-step GRPO smoke test
This confirms that model loading, 4-bit QLoRA, generation, rewards, backpropagation, and adapter saving work together. It is not the final experiment.

In [ ]:
!python train_grpo.py \
  --model Qwen/Qwen2.5-Coder-0.5B-Instruct \
  --train-data data/humaneval_train.jsonl \
  --eval-data data/humaneval_validation.jsonl \
  --quantization 4bit \
  --num-generations 2 \
  --gradient-accumulation-steps 2 \
  --max-completion-length 128 \
  --max-steps 1 \
  --executor local \
  --allow-local-code-execution \
  --output-dir outputs/qwen-code-grpo-smoke

## 8. Main training run
Keep the switch at `False` until the smoke test succeeds. Start with 10 steps; increase to 50 only after checking runtime and GPU memory.

In [ ]:
RUN_MAIN_TRAINING = False
MAIN_STEPS = 10

if RUN_MAIN_TRAINING:
    command = [
        'python', 'train_grpo.py',
        '--model', 'Qwen/Qwen2.5-Coder-0.5B-Instruct',
        '--train-data', 'data/humaneval_train.jsonl',
        '--eval-data', 'data/humaneval_validation.jsonl',
        '--quantization', '4bit',
        '--num-generations', '4',
        '--gradient-accumulation-steps', '4',
        '--max-completion-length', '256',
        '--max-steps', str(MAIN_STEPS),
        '--executor', 'local',
        '--allow-local-code-execution',
        '--output-dir', 'outputs/qwen-code-grpo',
    ]
    subprocess.run(command, check=True)
else:
    print('Main training is disabled. Set RUN_MAIN_TRAINING = True after the smoke test succeeds.')

## 9. Evaluate the trained adapter
This cell uses the main adapter when available; otherwise it evaluates the one-step smoke adapter only to verify the loading path.

In [ ]:
adapter = Path('outputs/qwen-code-grpo')
if not adapter.exists():
    adapter = Path('outputs/qwen-code-grpo-smoke')
print('Evaluating adapter:', adapter)
command = [
    'python', 'evaluate_baseline.py',
    '--data', 'data/humaneval_test.jsonl',
    '--backend', 'transformers',
    '--model', 'Qwen/Qwen2.5-Coder-0.5B-Instruct',
    '--adapter', str(adapter),
    '--quantization', '4bit',
    '--samples-per-task', '1',
    '--temperature', '0',
    '--executor', 'local',
    '--allow-local-code-execution',
    '--output-dir', 'results/grpo-heldout',
]
subprocess.run(command, check=True)

## 10. Compare and download results

In [ ]:
import json


def load_summary(path):
    return json.loads(Path(path).read_text())

baseline = load_summary('results/baseline-heldout/summary.json')
trained = load_summary('results/grpo-heldout/summary.json')
before = baseline['metrics']['pass@1']
after = trained['metrics']['pass@1']
print(f'Baseline pass@1: {before:.4f}')
print(f'GRPO pass@1:     {after:.4f}')
print(f'Difference:      {after - before:+.4f}')

In [ ]:
import shutil

from google.colab import files

bundle = Path('/content/code-grpo-experiment')
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir()
shutil.copytree('results', bundle / 'results')
shutil.copytree(adapter, bundle / 'adapter')
shutil.copy('data/manifest.json', bundle / 'data_manifest.json')
archive = shutil.make_archive('/content/code-grpo-experiment', 'zip', bundle)
print('Archive:', archive)
files.download(archive)